# 04 · Lista de casa: Polars e DuckDB

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

## Instruções

- Trabalhe em uma cópia deste notebook e substitua cada `...` pelo seu código.
- Nas questões de Polars, não converta para pandas; nas de SQL, a lógica deve estar na consulta.
- As células ✅ comparam o seu resultado com uma referência.
- Antes de entregar, use **Reiniciar sessão e executar tudo**.

| Questão | Tema | Pontos |
|---|---|---|
| 1 | Traduzir pandas para Polars | 2,0 |
| 2 | Traduzir para SQL | 2,0 |
| 3 | Plano de consulta e desempenho | 2,0 |
| 4 | Janela no tempo: a avaliação anterior do cliente | 2,5 |
| 5 | Escolha da ferramenta | 1,5 |

In [ ]:
import timeit
from pathlib import Path
from urllib.request import urlretrieve

import duckdb
import numpy as np
import pandas as pd
import polars as pl

print("polars", pl.__version__, "· duckdb", duckdb.__version__)

URL = (
    "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/"
    "4639429ec698d7821fc99a0bc665fa213d9fcd5a/B2W-Reviews01.csv"
)
ARQUIVO = Path("B2W-Reviews01.csv")
if not ARQUIVO.exists():
    urlretrieve(URL, ARQUIVO)

avaliacoes = (
    pl.read_csv(ARQUIVO, schema_overrides={"product_id": pl.String}, try_parse_dates=True)
    .with_columns(
        nota_5=(pl.col("overall_rating") == 5).cast(pl.Int8),
        caracteres=pl.col("review_text").str.len_chars(),
    )
    .drop("review_title", "review_text", "product_name")
)
avaliacoes_pd = avaliacoes.to_pandas()
avaliacoes.write_parquet("casa04.parquet")
print(avaliacoes.shape)

## Questão 1 (2,0 pontos): de pandas para Polars

Reescreva em Polars as três consultas abaixo, dadas em pandas, guardando os resultados em `q1a`, `q1b` e `q1c`. Use `avaliacoes` como fonte.

In [ ]:
# Referências em pandas (não altere)
ref_a = (
    avaliacoes_pd[avaliacoes_pd["reviewer_gender"].notna()]
    .groupby("reviewer_gender")
    .agg(n=("overall_rating", "size"), nota_media=("overall_rating", "mean"), pct_nota_5=("nota_5", "mean"))
    .reset_index()
    .sort_values("reviewer_gender")
)

ref_b = (
    avaliacoes_pd.assign(mes=avaliacoes_pd["submission_date"].dt.month)
    .query("site_category_lv1 == 'Livros'")
    .groupby("mes")
    .agg(n=("overall_rating", "size"), caracteres_mediana=("caracteres", "median"))
    .reset_index()
    .sort_values("mes")
)

ref_c = (
    avaliacoes_pd.assign(media_estado=avaliacoes_pd.groupby("reviewer_state")["overall_rating"].transform("mean"))
    .dropna(subset=["reviewer_state"])
    .query("overall_rating < media_estado - 2")
    .groupby("reviewer_state")
    .size()
    .rename("muito_abaixo")
    .reset_index()
    .sort_values("muito_abaixo", ascending=False)
    .head(5)
)
ref_a

In [ ]:
# ✏️ Questão 1
# SEU CÓDIGO AQUI
q1a = ...
q1b = ...
q1c = ...

In [ ]:
# ✅ Conferência da questão 1 (não altere)
def _igual(resultado, referencia):
    assert isinstance(resultado, pl.DataFrame), "o resultado deve ser um DataFrame do Polars"
    assert resultado.columns == list(referencia.columns), f"colunas esperadas: {list(referencia.columns)}"
    for coluna in referencia.columns:
        a, b = resultado[coluna].to_list(), referencia[coluna].tolist()
        assert len(a) == len(b), f"número de linhas diferente em {coluna}"
        assert all((x == y) or np.isclose(float(x), float(y)) for x, y in zip(a, b)) if referencia[coluna].dtype.kind in "if" else a == b, f"valores diferentes em {coluna}"

_igual(q1a, ref_a); _igual(q1b, ref_b); _igual(q1c, ref_c)
print("As três consultas coincidem com as referências em pandas.")

**1b)** Na consulta (a), a versão em pandas precisou de `notna()` antes do `groupby`? E a versão em Polars, precisa do filtro de ausentes? Explique a diferença de comportamento entre as duas bibliotecas.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 2 (2,0 pontos): as mesmas consultas em SQL

Escreva as consultas (a), (b) e (c) da questão 1 em SQL, nas variáveis `sql_a`, `sql_b` e `sql_c`, usando o DataFrame `avaliacoes` como fonte. As colunas e a ordem das linhas devem ser as mesmas das referências.

Dicas: `MONTH(coluna)` extrai o mês; `MEDIAN(coluna)` calcula a mediana; para (c), use uma etapa `WITH` com `AVG(...) OVER (PARTITION BY ...)`.

In [ ]:
# ✏️ Questão 2
# SEU CÓDIGO AQUI
sql_a = """ ... """
sql_b = """ ... """
sql_c = """ ... """

In [ ]:
# ✅ Conferência da questão 2 (não altere)
for _sql, _ref in [(sql_a, ref_a), (sql_b, ref_b), (sql_c, ref_c)]:
    _igual(duckdb.sql(_sql).pl(), _ref)
print("As três consultas em SQL coincidem com as referências.")

**2b)** Na consulta (c), por que a condição `overall_rating < media_estado - 2` não pode ficar no `WHERE` da mesma etapa em que a média por estado é calculada?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 3 (2,0 pontos): plano de consulta e desempenho

O arquivo `casa04.parquet`, criado na preparação, contém a tabela `avaliacoes`.

a) Escreva `consulta_q3`, uma consulta **preguiçosa** sobre `casa04.parquet` que devolva, para as avaliações de **maio** com nota 1 ou 2, o número de avaliações por `site_category_lv1` (coluna `n`), em ordem decrescente.

b) Guarde em `plano` o texto do plano otimizado (`explain()`), e em `colunas_lidas` o número de colunas que o plano indica que serão lidas do arquivo.

c) Meça, em `t_preguicoso` e `t_pandas`, o tempo da sua consulta e o da versão equivalente em pandas lendo o arquivo inteiro com `pd.read_parquet`.

In [ ]:
# ✏️ Questão 3
# SEU CÓDIGO AQUI
consulta_q3 = ...
plano = ...
colunas_lidas = ...
t_preguicoso = ...
t_pandas = ...

In [ ]:
# ✅ Conferência da questão 3 (não altere)
assert isinstance(consulta_q3, pl.LazyFrame), "a consulta deve ser preguiçosa"
_r = consulta_q3.collect()
_refq3 = avaliacoes_pd[(avaliacoes_pd["submission_date"].dt.month == 5) & (avaliacoes_pd["overall_rating"] <= 2)].groupby("site_category_lv1").size().sort_values(ascending=False)
assert _r.columns == ["site_category_lv1", "n"] and _r["n"].to_list() == _refq3.tolist(), "o resultado difere do esperado"
assert isinstance(plano, str) and len(plano) > 50 and colunas_lidas == 3
assert t_preguicoso > 0 and t_pandas > 0
print("Formato correto.")

**3b)** Com base no plano: quais colunas são lidas e onde o filtro é aplicado? Dê os nomes das duas otimizações. Depois comente a razão entre os tempos: por que, neste arquivo, ela é menor do que a obtida em aula com o arquivo de 2,6 milhões de linhas?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 4 (2,5 pontos): a avaliação anterior do cliente

Queremos saber se a nota que um cliente dá está relacionada com a nota que ele deu na **avaliação anterior**.

a) Em Polars, construa `pares`: uma linha para cada avaliação que **tem** uma anterior do mesmo cliente, com as colunas `reviewer_id`, `submission_date`, `overall_rating` e `nota_anterior`. Dica: ordene por data e use `pl.col("overall_rating").shift(1).over("reviewer_id")`.

b) Em SQL, calcule `tabela_q4`: para cada valor de `nota_anterior` (de 1 a 5), o número de pares (`n`) e a proporção deles em que a nota atual é 4 ou 5 (`pct_positiva`), ordenada por `nota_anterior`. Dica: `LAG(overall_rating) OVER (PARTITION BY reviewer_id ORDER BY submission_date)`.

In [ ]:
# ✏️ Questão 4
# SEU CÓDIGO AQUI
pares = ...
tabela_q4 = ...

In [ ]:
# ✅ Conferência da questão 4 (não altere)
_esperado = len(avaliacoes_pd) - avaliacoes_pd["reviewer_id"].nunique()
assert pares.columns == ["reviewer_id", "submission_date", "overall_rating", "nota_anterior"]
assert abs(pares.height - _esperado) <= 0, f"eram esperados {_esperado} pares (avaliações menos clientes distintos)"
assert tabela_q4.columns == ["nota_anterior", "n", "pct_positiva"] and tabela_q4.height == 5
assert tabela_q4["n"].sum() == pares.height and tabela_q4["nota_anterior"].to_list() == [1, 2, 3, 4, 5]
print(f"Formato correto: {pares.height} pares. Proporção geral de notas 4 e 5: {(avaliacoes['overall_rating'] >= 4).mean():.3f}")

**4b)** O que a tabela mostra? Compare `pct_positiva` de quem deu nota 1 antes com a de quem deu nota 5 antes, e com a proporção geral. Proponha duas explicações diferentes para o padrão e diga por que esse resultado importa para a divisão dos dados em treino e teste.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 5 (1,5 ponto): que ferramenta você usaria?

Para cada situação, escolha entre pandas, Polars e DuckDB e justifique em duas frases. Declare ao final como usou assistentes de IA nesta lista.

1. Uma planilha de 40 mil linhas com o resultado de uma pesquisa, para uma análise exploratória de uma tarde.
2. Três anos de registros de acesso em 600 arquivos Parquet, 80 GB no total, dos quais você precisa de totais diários por serviço.
3. Um processo que roda toda noite, lê 15 GB de CSV, aplica vinte transformações encadeadas e grava o resultado.
4. Uma biblioteca de séries temporais que só aceita DataFrames do pandas, aplicada a uma tabela de 2 milhões de linhas obtida pela junção de quatro tabelas grandes.

✏️ **Sua resposta:**

_Escreva aqui._